## Text input

In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model='gpt-5-nano',
    system_prompt="Eres escritor de ciencia ficción, crea una capital a petición de los usuarios."
)

In [ ]:
from langchain.messages import HumanMessage

question = HumanMessage(content=[
    {"type": "text", "text": "¿Cuál es la capital de la Luna?"}
])

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

## Image input

https://platform.openai.com/docs/models

In [ ]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

In [ ]:
print(uploader.value)

In [ ]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [ ]:
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Cuéntame sobre esta capital"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

## Audio input

In [ ]:
import base64
import io
import time
import numpy as np
from scipy.io.wavfile import write
import sounddevice as sd
from tqdm import tqdm

# Configuración recomendada para APIs de voz
duration = 5  # segundos
sample_rate = 24000  # 16000 o 24000 Hz es el estándar nativo de OpenAI

print("Grabando... habla al micrófono:")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1, dtype='float32')

for _ in tqdm(range(duration * 10)):
    time.sleep(0.1)
sd.wait()
print("Completado.")

# 1. Comprobar si realmente capturó sonido (si da ~0.0, falta permiso de micrófono en macOS)
max_amp = np.abs(audio).max()
print(f"Amplitud máxima capturada: {max_amp:.4f}")
if max_amp < 0.01:
    print("⚠️ Advertencia: El audio grabado está prácticamente en silencio. Revisa los permisos de micrófono en macOS o sd.default.device.")

# 2. Convertir float32 [-1.0, 1.0] a PCM 16-bit int16 [-32768, 32767]
audio_int16 = (np.clip(audio, -1.0, 1.0) * 32767).astype(np.int16)

# 3. Escribir WAV en memoria
buf = io.BytesIO()
write(buf, sample_rate, audio_int16)
wav_bytes = buf.getvalue()

aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")


In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(
    model="openai:gpt-audio-1.5",
)

multimodal_question = HumanMessage(
    content=[
        {"type": "text", "text": "Describe el mensaje del audio"},
        {
            "type": "audio",
            "base64": aud_b64,
            "mime_type": "audio/wav",
        },
    ]
)

response = agent.invoke({"messages": [multimodal_question]})
print(response["messages"][-1].content)


In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

system_prompt = (
    "Eres un asistente especializado en procesamiento y análisis de audio. "
    "Tu tarea es transcribir lo que escuchas con precisión y luego resumir los puntos clave."
)

agent = create_agent(
    model="openai:gpt-audio-1.5",
    system_prompt=system_prompt,
)

multimodal_question = HumanMessage(
    content=[
        {"type": "text", "text": "Describe el mensaje del siguiente audio"},
        {
            "type": "audio",
            "base64": aud_b64,
            "mime_type": "audio/wav",
        },
    ]
)

response = agent.invoke({"messages": [multimodal_question]})
print(response["messages"][-1].content)
